# Notebook 2: SFT Training with DoRA
**Platform: Kaggle (2x T4 GPUs — 32GB VRAM total)**

## What this notebook does:
1. Loads cleaned dataset from Google Drive
2. Downloads LLaMA 2 7B Chat from HuggingFace
3. Fine-tunes with DoRA (deeper than LoRA)
4. Saves checkpoints to HuggingFace Hub automatically
5. Pushes final SFT adapter to HuggingFace Hub

## Kaggle setup before running:
1. New notebook > Settings > Accelerator > **GPU T4 x2**
2. Settings > Internet > **ON**
3. Settings > Secrets > Add:
   - Name: `HF_TOKEN` | Value: your HuggingFace token

## If session expires mid-training:
Set `RESUME = True` below and re-run all cells.


In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 13.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.6 MB/s eta 0:00:00:00:0100:01


In [2]:
import os, json, torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets    = UserSecretsClient()
HF_TOKEN   = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN)

# REPLACE with your HuggingFace username
HF_USERNAME = 'Winindux'

MODEL_NAME = 'meta-llama/Llama-2-7b-chat-hf'  # your approved model
SFT_REPO   = f'{HF_USERNAME}/emowoz-llama2-sft'
RESUME     = False   # set True to continue from last checkpoint

print(f'Base model : {MODEL_NAME}')
print(f'SFT output : {SFT_REPO}')
print(f'GPUs       : {torch.cuda.device_count()}')

Base model : meta-llama/Llama-2-7b-chat-hf
SFT output : Winindux/emowoz-llama2-sft
GPUs       : 2


In [3]:
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient
from datasets import Dataset
import json, os

secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN)

HF_USERNAME = 'Winindux'
DATA_REPO   = f'{HF_USERNAME}/emowoz-llama2-data'

def download(hf_path):
    return hf_hub_download(
        repo_id=DATA_REPO,
        filename=hf_path,
        repo_type='dataset',
        local_dir='/kaggle/working',
        token=HF_TOKEN
    )

train_path = download('data/train/sft_train.jsonl')
val_path   = download('data/val/sft_val.jsonl')
print('Data downloaded from HuggingFace Hub.')

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

train_raw = load_jsonl(train_path)
val_raw   = load_jsonl(val_path)

train_ds = Dataset.from_list([{'text': ex['text']} for ex in train_raw])
val_ds   = Dataset.from_list([{'text': ex['text']} for ex in val_raw])

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print('\nSample (first 400 chars):')
print(train_ds[0]['text'][:400])

sft_train.jsonl:   0%|          | 0.00/9.12k [00:00<?, ?B/s]

sft_val.jsonl:   0%|          | 0.00/696 [00:00<?, ?B/s]

Data downloaded from HuggingFace Hub.
Train: 8 | Val: 1

Sample (first 400 chars):
<s>[INST] <<SYS>>
You are a compassionate and helpful assistant. Follow these rules strictly:
1. Respond using ONLY the information provided in the Facts section.
2. Never add information that is not present in the Facts.
3. Adapt your tone to match the user's emotional state.
4. Be warm and empathetic while remaining factually accurate.
<</SYS>>

Emotional State: satisfied
Facts: Restaurant postc


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit quantization: shrinks LLaMA 2 7B from ~14GB to ~4GB VRAM
# Essential to fit on Kaggle 2xT4 (32GB total)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,  # extra compression
    bnb_4bit_quant_type='nf4',       # best quality quant type for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f'Loading tokenizer for {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Loading model (4-bit quantized)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',        # spread across both T4 GPUs automatically
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16
)

print('Model loaded.')
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {used:.1f} / {total:.1f} GB used')

Loading tokenizer for meta-llama/Llama-2-7b-chat-hf...


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Model loaded.
  GPU 0: 1.5 / 15.6 GB used
  GPU 1: 2.4 / 15.6 GB used


In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# DoRA config — use_dora=True is the key difference from LoRA
# We target all major projection layers including FFN (gate/up/down_proj)
# LoRA typically only targets q_proj + v_proj; DoRA covers everything
dora_config = LoraConfig(
    r=64,              # rank — higher = more expressive adaptation
    lora_alpha=128,    # scale = 2x rank is standard
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',   # attention layers
        'gate_proj', 'up_proj', 'down_proj'        # feed-forward layers
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    use_dora=True      # THIS makes it DoRA, not LoRA
)

model = get_peft_model(model, dora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)')

Trainable: 161.3M / 3661.7M (4.40%)


In [6]:
# from trl import SFTConfig, SFTTrainer

# OUTPUT_DIR = '/kaggle/working/sft_checkpoints'
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# training_args = SFTConfig(
#     output_dir=OUTPUT_DIR,

#     # Training
#     num_train_epochs=3,
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     gradient_accumulation_steps=8,   # effective batch = 2 * 2 GPUs * 8 = 32

#     # Memory optimisations
#     gradient_checkpointing=True,     # saves ~30% VRAM, ~20% slower
#     optim='paged_adamw_8bit',        # 8-bit optimizer — big memory saving
#     bf16=True,
#     fp16=False,

#     # Learning rate
#     learning_rate=2e-4,
#     lr_scheduler_type='cosine',
#     warmup_ratio=0.05,
#     max_grad_norm=0.3,
#     weight_decay=0.001,

#     # Sequence
#     max_seq_length=1024,
#     packing=True,                    # packs short sequences together — faster training

#     # Evaluation
#     eval_strategy='steps',
#     eval_steps=100,

#     # Checkpointing — CRITICAL for session recovery
#     save_strategy='steps',
#     save_steps=200,
#     save_total_limit=3,              # keep last 3 only to save disk
#     load_best_model_at_end=True,
#     metric_for_best_model='eval_loss',

#     # Auto-push every checkpoint to HuggingFace Hub
#     push_to_hub=True,
#     hub_model_id=SFT_REPO,
#     hub_strategy='checkpoint',
#     hub_token=HF_TOKEN,

#     logging_steps=50,
#     report_to='tensorboard',
# )

In [7]:
# trainer = SFTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_ds,
#     eval_dataset=val_ds,
#     tokenizer=tokenizer,
# )

# if RESUME:
#     print('Resuming from last HF Hub checkpoint...')
#     trainer.train(resume_from_checkpoint=True)
# else:
#     print('Starting SFT training from scratch...')
#     trainer.train()

# print('SFT training complete!')

In [8]:
# Cell 6 — training config
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = '/kaggle/working/sft_checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Tokenize the dataset once upfront — avoids all SFTTrainer parameter drama
def tokenize(example):
    return tokenizer(
        example['text'],
        truncation=True,
        max_length=1024,
        padding=False,
    )

train_tokenized = train_ds.map(tokenize, remove_columns=['text'])
val_tokenized   = val_ds.map(tokenize,   remove_columns=['text'])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # causal LM, not masked
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Memory
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    bf16=True,
    fp16=False,

    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    max_grad_norm=0.3,
    weight_decay=0.001,

    # Evaluation
    eval_strategy='steps',
    eval_steps=100,

    # Checkpointing
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    # Push to HF Hub
    push_to_hub=True,
    hub_model_id=SFT_REPO,
    hub_strategy='checkpoint',
    hub_token=HF_TOKEN,

    logging_steps=50,
    report_to='tensorboard',
)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [9]:
# Cell 7 — trainer + run
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

if RESUME:
    print('Resuming from last checkpoint...')
    trainer.train(resume_from_checkpoint=True)
else:
    print('Starting SFT training...')
    trainer.train()

print('SFT training complete!')

2026-02-22 12:30:11.577837: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771763411.792390      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771763411.866129      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771763412.412517      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771763412.412549      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771763412.412552      55 computation_placer.cc:177] computation placer alr

Starting SFT training...


Step,Training Loss,Validation Loss


SFT training complete!


In [10]:
# Save only the DoRA adapter (~200-400MB, NOT the full 14GB model)
# At inference time: load base model from HF + apply this adapter
ADAPTER_DIR = '/kaggle/working/sft_final_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(SFT_REPO, private=True, exist_ok=True)
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=SFT_REPO,
    commit_message='Final SFT DoRA adapter — LLaMA 2 7B EmoWOZ'
)
print(f'Adapter saved to: https://huggingface.co/{SFT_REPO}')

# Quick sanity check — generate one response
test_prompt = (
    "<s>[INST] <<SYS>>\n"
    "You are a compassionate assistant. Use ONLY the provided Facts. Match the user's emotional tone.\n"
    "<</SYS>>\n\n"
    "Emotional State: anxious\n"
    "Facts: The Gonville Hotel is in the centre of Cambridge. Price: expensive. Phone: 01223366611.\n"
    "Conversation History:\nNo previous turns.\n"
    "User: I need a hotel but I'm really stressed about my trip [/INST]"
)
inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=120, temperature=0.7,
                     do_sample=True, pad_token_id=tokenizer.eos_token_id,
                     repetition_penalty=1.3)
print('\nTest response:')
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


Adapter saved to: https://huggingface.co/Winindux/emowoz-llama2-sft

Test response:
 sierp hopefully hopefully hopefully Begriffe obviously hopefully Begriffe фев everybody obviouslyϊ everybody живело➖ surely nobody nobody everybody Hinweis nobodyℓ фев everyone everybody➖ surely obviously Einzeln Hinweis живело фев nobody hopefully nobody nobody everyone фев фев Hinweis Unterscheidung sierp everybody everybody nobody everybody Hinweis hopefully Hinweis Hinweis Einzeln hopefully hopefully everyone Hinweis Einzeln nobody Unterscheidung округу hopefully фев();` Bedeut Hinweis everybody Unterscheidung everybody nobody Hinweis✿ nobody obviously everybody Einzeln hopefully ultimately hopefully Hinweis paździer фев hopefully Einzeln everybody hopefully everybody obviously hopefully Unterscheidung nobody➖ nobody nobody everybody Unterscheidung Unterscheidung Unterscheidung nobody nobody Hinweisℓ hopefully nobody hopefully червня Einzeln nobody Begriffe Unterscheidung hopefully Hinweis nobody n

## ✅ Notebook 2 Complete!
SFT adapter saved at: `huggingface.co/YOUR_USERNAME/emowoz-llama2-sft`

## ➡️ Next: Create a NEW Kaggle notebook and run Notebook 3 (DPO)